# Lesson 0.3: Population vs Sample

See that a sample statistic wanders around the fixed population parameter, that bigger samples wander less, and that a biased sample does not.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A university has 40,000 students and wants their average weekly study time, but it can only ask a few. Before trusting any sample, we must see how a sample mean behaves next to the true population mean. We start with a population small enough to see (ten shops' sales) and then use 10,000 simulated study times.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: The lesson's ten shops

Here the population is small enough to see: monthly sales of 10 shops. The population mean (a parameter) is fixed. Each sample of 3 shops gives a different sample mean (a statistic). The population SD divides by N (ddof=0), the sample SD by n - 1 (ddof=1).

In [1]:
import numpy as np  # Load NumPy, the library for lists of numbers, as np.

shops = np.array([12, 15, 9, 22, 18, 7, 25, 14, 20, 11])  # Store the whole population: the monthly sales (in thousands) of all ten shops, so the truth is known exactly.
mu = shops.mean()  # mu is the population mean. It is a parameter: one fixed number that never changes.
print(f'population mean (mu) = {mu:.2f}, sigma = {shops.std(ddof=0):.3f}')  # Print mu and sigma. ddof=0 divides by N, which is right because this is the whole population and not a sample.

samples = {'A': shops[[0, 3, 5]], 'B': shops[[6, 4, 1]], 'C': shops[[2, 9, 7]]}  # Three samples of 3 shops each, picked by position. Python counts positions from 0.
for name, s in samples.items():  # Loop over the samples, taking the name and the values of each.
    print(f'sample {name}: {s}  x-bar = {s.mean():.2f}  gap = {s.mean() - mu:+.2f}')  # Show each sample's mean (a statistic) and its gap from mu. That gap is the sampling error.
print(f'sample A sd = {samples["A"].std(ddof=1):.3f}')  # Sample standard deviation with ddof=1: dividing by n - 1 makes up for a sample under-estimating the spread of the population.

population mean (mu) = 15.30, sigma = 5.551
sample A: [12 22  7]  x-bar = 13.67  gap = -1.63
sample B: [25 18 15]  x-bar = 19.33  gap = +4.03
sample C: [ 9 11 14]  x-bar = 11.33  gap = -3.97
sample A sd = 7.638


**What this means.** The population mean mu is 15.30 and it never moves. The three samples give 13.67, 19.33 and 11.33, which are gaps of -1.63, +4.03 and -3.97. Each sample is honest, yet none hits 15.30. That miss is sampling error and it is normal. With only three shops, sample A's SD (7.638) can also be far from sigma (5.551).

## Step 2: A bigger, seeded population

Now a population of 10,000 weekly study times in hours, made with a fixed seed so everyone gets the same numbers. We pretend not to know its mean and compute it once: this is the parameter.

In [2]:
rng = np.random.default_rng(42)  # Create a random number generator with seed 42. A fixed seed gives everyone the same 'random' numbers.
population = rng.normal(loc=14, scale=4, size=10_000).round(1)  # Make 10,000 weekly study times from a bell curve with mean 14 and SD 4, rounded to 0.1 hour. This is our pretend population.

mu = population.mean()  # Compute the population mean once. In real life it is unknown, here it is the target our samples aim at.
print('population size N =', len(population))  # Print the population size N.
print(f'population mean mu = {mu:.3f}')  # Print the population mean, our parameter.

population size N = 10000
population mean mu = 13.959


**What this means.** The population holds 10,000 study times with a true mean of 13.959 hours. That is close to the 14 we aimed for, but not exactly 14, because even the population is one random draw. This fixed number is the parameter that every sample will try to hit.

## Step 3: Sampling error shrinks as n grows

For each sample size we draw one random sample and show its gap from mu. Then we repeat the draw 300 times to see the typical gap. Bigger samples miss by less. The gap never vanishes completely: that is sampling error, and it is normal.

In [3]:
for n in [10, 50, 250, 1000]:  # Try four sample sizes, from tiny to large.
    one = rng.choice(population, size=n, replace=False).mean()  # Draw one random sample without replacement (nobody asked twice) and take its mean.
    many = [rng.choice(population, size=n, replace=False).mean() for _ in range(300)]  # Repeat the draw 300 times to see the pattern of sample means, not just one lucky or unlucky sample.
    typical = np.mean(np.abs(np.array(many) - mu))  # Average the absolute gaps between those 300 sample means and mu. That is the typical sampling error.
    print(f'n = {n:4d}: one sample mean = {one:6.2f}, typical gap over 300 samples = {typical:.3f}')  # Print the size, the one sample's mean and the typical gap.

n =   10: one sample mean =  12.40, typical gap over 300 samples = 1.066
n =   50: one sample mean =  14.58, typical gap over 300 samples = 0.473
n =  250: one sample mean =  14.04, typical gap over 300 samples = 0.203
n = 1000: one sample mean =  13.80, typical gap over 300 samples = 0.098


**What this means.** The single sample means (12.40, 14.58, 14.04, 13.80) wander around 13.959. The typical gap shrinks as n grows: 1.066 at n = 10, 0.473 at 50, 0.203 at 250 and 0.098 at 1000. Using 25 times as many students cuts the error about five-fold. Bigger samples miss by less, but never by exactly zero.

## Step 4: A biased sample does not get better

Suppose we only ask the 1,000 heaviest studiers (say, people we meet in the library). Even with a large sample the mean stays far from mu, because the method is tilted. A bigger sample cannot fix bias.

In [4]:
top = np.sort(population)[-1000:]  # Sort the study times and keep the 1,000 highest: only the heaviest studiers, like asking only people in the library.

for n in [10, 100, 1000]:  # Try three sample sizes, again from small to large.
    m = rng.choice(top, size=n, replace=False).mean()  # Draw a sample from that tilted group only, and take its mean.
    print(f'biased sample n = {n:4d}: mean = {m:.2f}, gap from mu = {m - mu:+.2f}')  # Print the mean and its gap from the true mu, with a plus or minus sign.

biased sample n =   10: mean = 21.05, gap from mu = +7.09
biased sample n =  100: mean = 20.79, gap from mu = +6.83
biased sample n = 1000: mean = 21.05, gap from mu = +7.09


**What this means.** All three means are around 21 hours, so the gaps from mu are +7.09, +6.83 and +7.09, whatever the sample size. A bigger sample shrinks random error but not bias, because the bias comes from who we ask. Choose fairly first, then make the sample large.

## So, what is the answer?

A sample statistic is not the population parameter: it wanders around it by sampling error, as our three shop samples (13.67, 19.33, 11.33 against mu = 15.30) showed. Bigger samples wander less (typical gap 1.07 at n = 10 and 0.10 at n = 1000), but a biased way of choosing, such as asking only the heaviest studiers, stays about 7 hours off at any size.

## Now you try

A landlord owns eight flats and knows the monthly electricity bill (in dollars) of every one. He wonders how well the mean of a small sample of flats would estimate the true mean of all eight, and whether asking more flats helps.

**Data:** bills = [52, 61, 48, 75, 66, 58, 90, 54]

1. Find the population mean and the population standard deviation (ddof=0).
2. List every possible sample of 2 flats and of 5 flats (itertools.combinations) and compute each sample mean.
3. Compare the typical gap from mu for the two sample sizes, and the average of all sample means with mu.

*Hint:* itertools.combinations(bills, n) gives every way to pick n flats. Take np.mean of each, then compare np.abs(means - mu).mean().

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from itertools import combinations

bills = np.array([52, 61, 48, 75, 66, 58, 90, 54])
mu = bills.mean()
print(f'mu = {mu:.2f}, sigma = {bills.std(ddof=0):.2f}')
for n in [2, 5]:
    means = np.array([np.mean(c) for c in combinations(bills, n)])
    print(f'n = {n}: {len(means)} samples, average of sample means = {means.mean():.2f}, '
          f'typical gap = {np.abs(means - mu).mean():.2f}, biggest gap = {np.abs(means - mu).max():.2f}')

mu = 63.00, sigma = 12.93
n = 2: 28 samples, average of sample means = 63.00, typical gap = 7.11, biggest gap = 19.50
n = 5: 56 samples, average of sample means = 63.00, typical gap = 3.17, biggest gap = 8.40


**Answer.** mu = 63.00 and sigma = 12.93. Samples of 2 give 28 possible means with a typical gap of 7.11 (the worst is 19.50). Samples of 5 give 56 means with a typical gap of 3.17 (the worst is 8.40). All sample means average exactly 63.00, so sampling is not biased, and bigger samples land closer to mu.

## The same question in R, SPSS and Stata

**R**

```r
shops <- c(12, 15, 9, 22, 18, 7, 25, 14, 20, 11)  # Store the population: the sales of all ten shops, in a vector.
mean(shops)  # mean(shops) is the population mean mu.
sqrt(mean((shops - mean(shops))^2))  # R's sd() divides by n - 1, so for sigma we average the squared gaps ourselves (dividing by N) and take the square root.
A <- shops[c(1, 4, 6)]  # Sample A: shops number 1, 4 and 6. R counts positions from 1.
mean(A)  # The mean of sample A is a statistic.
sd(A)  # sd() of sample A is the sample standard deviation (n - 1), right for a sample.
mean(shops[c(7, 5, 2)])  # Sample B: shops 7, 5 and 2, and its mean.
mean(shops[c(3, 10, 8)])  # Sample C: shops 3, 10 and 8, and its mean.
```

How to read it: R prints mu = 15.3 and sigma = 5.551, then sample A's mean 13.67 and SD 7.638, then 19.33 for sample B and 11.33 for sample C. The three statistics differ from each other and from 15.3, while mu stays put.

**SPSS** (Analyze > Descriptive Statistics > Descriptives)

```
DATA LIST FREE / sales inA.
BEGIN DATA
12 1 15 0 9 0 22 1 18 0
7 1 25 0 14 0 20 0 11 0
END DATA.
DESCRIPTIVES VARIABLES=sales.
TEMPORARY.
SELECT IF (inA = 1).
DESCRIPTIVES VARIABLES=sales.
```

How to read it: The first Descriptives table (N = 10) shows a Mean of 15.30, which is mu. Its Std. Deviation 5.851 is the sample version, not sigma 5.551. The second table (N = 3) shows Mean 13.67 and Std. Deviation 7.638: sample A's statistics.

**Stata**

```
clear
set obs 10
generate sales = real(word("12 15 9 22 18 7 25 14 20 11", _n))
generate inA = inlist(_n, 1, 4, 6)
summarize sales
summarize sales if inA == 1
```

How to read it: The first summarize shows 10 observations and a Mean of 15.3 (mu). Its Std. Dev. 5.851 uses n - 1. The second shows Obs 3, Mean 13.67 and Std. Dev. 7.638, the statistics of sample A, which differ from mu.